# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mfaiqdev/MLinternship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The action queue prioritizes content for human review using the validated Week-4 baseline and supporting performance signals.

The primary reason code is `HIGH_IMPRESSIONS_LOW_CLICKS`, based on the observed rule of more than 100 impressions and no more than 1 click. This identifies content with search visibility but weak click activity.

Other reason codes are used only as supporting review signals. They do not mean that content definitely requires a refresh.

The queue uses simple performance archetypes to connect observed signals to recommended actions. These archetypes describe performance patterns; they are not content-quality or topic labels.

The queue is ranked for review priority, not as a probability of refresh success. The purpose is to help a reviewer decide what to inspect first and why.

In [1]:
import os
import numpy as np
import pandas as pd

# Load the same March 2026 development data used in Weeks 4–6.
df = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

# Aggregate the daily panel to one row per content item.
content_df = (
    df.groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        total_impressions=("gsc_impressions", "sum"),
        total_clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        total_pageviews=("ga4_pageviews", "sum"),
        total_users=("ga4_users", "sum"),
        total_engagement_sec=("ga4_total_engagement_sec", "sum"),
        total_scroll_events=("scroll_events", "sum"),
        active_days=("report_date", "nunique")
    )
)

# Observed CTR.
content_df["observed_ctr"] = np.where(
    content_df["total_impressions"] > 0,
    content_df["total_clicks"] / content_df["total_impressions"],
    np.nan
)

# Week-4 baseline signal.
content_df["baseline_opportunity"] = (
    (content_df["total_impressions"] > 100)
    & (content_df["total_clicks"] <= 1)
)

# Assign reason codes.
content_df["reason_code"] = np.select(
    [
        content_df["baseline_opportunity"],
        (
            (content_df["total_impressions"] > 100)
            & (content_df["observed_ctr"] < 0.05)
        ),
        (
            content_df["avg_position"].notna()
            & (content_df["avg_position"] > 20)
        )
    ],
    [
        "HIGH_IMPRESSIONS_LOW_CLICKS",
        "VISIBLE_LOW_CTR",
        "WEAK_SEARCH_POSITION"
    ],
    default="MONITOR_ONLY"
)

# Assign performance archetypes.
content_df["archetype"] = np.select(
    [
        content_df["baseline_opportunity"],
        (
            (content_df["total_impressions"] > 100)
            & (content_df["observed_ctr"] < 0.05)
        ),
        (
            content_df["avg_position"].notna()
            & (content_df["avg_position"] > 20)
        ),
        (content_df["total_impressions"] <= 100)
    ],
    [
        "HIGH_VISIBILITY_LOW_CLICKS",
        "VISIBLE_LOW_CTR",
        "WEAK_RANKING",
        "LOW_EVIDENCE"
    ],
    default="MONITOR_ONLY"
)

# Archetype-to-action mapping.
action_map = {
    "HIGH_VISIBILITY_LOW_CLICKS":
        "Review title, snippet, search intent, and content relevance.",
    "VISIBLE_LOW_CTR":
        "Review SERP presentation and title/snippet relevance.",
    "WEAK_RANKING":
        "Review relevance, content depth, internal linking, and search intent.",
    "LOW_EVIDENCE":
        "Monitor and gather more evidence before investing in a refresh.",
    "MONITOR_ONLY":
        "Monitor; no refresh recommendation from this playbook alone."
}

content_df["recommended_action"] = content_df["archetype"].map(action_map)

# Transparent action-priority score.
impression_score = (
    np.log1p(content_df["total_impressions"])
    / np.log1p(content_df["total_impressions"]).quantile(0.95)
).clip(0, 1)

ctr_score = (
    1 - content_df["observed_ctr"].fillna(0)
).clip(0, 1)

# Missing ranking position is treated as no reliable ranking evidence.
# Filling with 50 prevents the priority score from becoming missing.
position_score = (
    (content_df["avg_position"].fillna(50) - 1) / 49
).clip(0, 1)

content_df["priority_score"] = (
    0.50 * impression_score
    + 0.30 * ctr_score
    + 0.20 * position_score
)

# Give baseline opportunities the highest review priority.
content_df.loc[
    content_df["baseline_opportunity"],
    "priority_score"
] += 0.25

# Priority tier.
content_df["priority"] = np.select(
    [
        content_df["baseline_opportunity"],
        content_df["reason_code"].isin(
            ["VISIBLE_LOW_CTR", "WEAK_SEARCH_POSITION"]
        )
    ],
    [
        "HIGH",
        "MEDIUM"
    ],
    default="LOW"
)

# Rank the queue.
priority_order = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}

content_df["priority_order"] = (
    content_df["priority"].map(priority_order)
)

action_queue = (
    content_df
    .sort_values(
        ["priority_order", "priority_score", "total_impressions"],
        ascending=[True, False, False]
    )
    .reset_index(drop=True)
)

action_queue["rank"] = np.arange(1, len(action_queue) + 1)

display(
    action_queue[
        [
            "rank",
            "content_hash_id",
            "priority",
            "reason_code",
            "archetype",
            "priority_score",
            "recommended_action"
        ]
    ].head(20)
)

print("Queue size:", len(action_queue))
print("High-priority items:", (action_queue["priority"] == "HIGH").sum())
print("Medium-priority items:", (action_queue["priority"] == "MEDIUM").sum())
print("Low-priority items:", (action_queue["priority"] == "LOW").sum())

,rank,content_hash_id,priority,reason_code,archetype,priority_score,recommended_action
0,1,content_295e883e0e86ca3c,HIGH,HIGH_IMPRESSIONS_LOW_CLICKS,HIGH_VISIBILITY_LOW_CLICKS,1.250000,"Review title, snippet, search intent, and cont..."
1,2,content_4002467a580a7f98,HIGH,HIGH_IMPRESSIONS_LOW_CLICKS,HIGH_VISIBILITY_LOW_CLICKS,1.250000,"Review title, snippet, search intent, and cont..."
2,3,content_066bb7aeff9aeea8,HIGH,HIGH_IMPRESSIONS_LOW_CLICKS,HIGH_VISIBILITY_LOW_CLICKS,1.250000,"Review title, snippet, search intent, and cont..."
3,4,content_959d535a9fcc865c,HIGH,HIGH_IMPRESSIONS_LOW_CLICKS,HIGH_VISIBILITY_LOW_CLICKS,1.250000,"Review title, snippet, search intent, and cont..."
4,5,content_6177aad2ded9dee5,HIGH,HIGH_IMPRESSIONS_LOW_CLICKS,HIGH_VISIBILITY_LOW_CLICKS,1.250000,"Review title, snippet, search intent, and cont..."
5,6,content_65b8a4998e633d89,HIGH,HIGH_IMPRESSIONS_LOW_CLICKS,HIGH_VISIBILITY_LOW_CLICKS,1.250000,"Review title, snippet, search intent, and cont..."
6,7,content_d8dfa985e6b95745,HIGH,HIGH_IMPRESSIONS_LOW_CLICKS,HIGH_VISIBILITY_LOW_CLICKS,1.250000,"Review title, snippet, search intent, and cont..."
7,8,content_92459e3cd3238cbf,HIGH,HIGH_IMPRESSIONS_LOW_CLICKS,HIGH_VISIBILITY_LOW_CLICKS,1.250000,"Review title, snippet, search intent, and cont..."
8,9,content_af748217e2ec86bc,HIGH,HIGH_IMPRESSIONS_LOW_CLICKS,HIGH_VISIBILITY_LOW_CLICKS,1.250000,"Review title, snippet, search intent, and cont..."
9,10,content_19c9547dd904c577,HIGH,HIGH_IMPRESSIONS_LOW_CLICKS,HIGH_VISIBILITY_LOW_CLICKS,1.250000,"Review title, snippet, search intent, and cont..."


Queue size: 331437
High-priority items: 53994
Medium-priority items: 68180
Low-priority items: 209263


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This playbook is intended for SEO or content teams that need to prioritize which content items should receive human review first.

The intended decision is:

> Which content should we inspect first for a possible refresh?

The playbook is not intended to determine automatically that a page needs a refresh.

The strongest evidence in this project is the Week-4 rule-based baseline. The Random Forest did not outperform that baseline, so the ML result is treated as directional supporting evidence rather than as the primary decision rule.

The refresh label used in Weeks 5–6 is a proxy constructed from performance conditions. It is not ground truth for whether a page actually needs editing.

The March 2026 data represent one development window. Performance can change over time, and the observed relationships may not hold for every client, content type, market, or future period.

The queue should therefore be used as decision-support. A human should verify the underlying page, search intent, business value, and reason for weak performance before taking action.

In [2]:
# Basic checks supporting the intended-use limits.

print("Content items in action queue:", len(action_queue))

print(
    "Baseline opportunities:",
    int(action_queue["baseline_opportunity"].sum())
)

print(
    "Baseline opportunity rate:",
    round(
        action_queue["baseline_opportunity"].mean() * 100,
        2
    ),
    "%"
)

print("\nPriority distribution:")

display(
    action_queue["priority"]
    .value_counts()
    .rename_axis("priority")
    .reset_index(name="content_items")
)

print("\nReason-code distribution:")

display(
    action_queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="content_items")
)

Content items in action queue: 331437
Baseline opportunities: 53994
Baseline opportunity rate: 16.29 %

Priority distribution:


,priority,content_items
0,LOW,209263
1,MEDIUM,68180
2,HIGH,53994



Reason-code distribution:


,reason_code,content_items
0,MONITOR_ONLY,209263
1,HIGH_IMPRESSIONS_LOW_CLICKS,53994
2,VISIBLE_LOW_CTR,47190
3,WEAK_SEARCH_POSITION,20990


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before taking action on a queue item, a human reviewer should check:

1. Whether the page is still relevant to its intended search intent.
2. Whether the information is outdated, incomplete, or inaccurate.
3. Whether the title and search-result presentation match the content.
4. Whether the observed performance issue is large enough to justify the expected editing effort.
5. Whether the page has business value or strategic importance.
6. Whether another explanation, such as SERP features or changes in search behaviour, could explain the observed performance.
7. Whether the proposed change can be made without creating factual, legal, brand, or user-experience risks.

Cost and value should also be considered. A high-visibility opportunity may justify more review effort than a low-visibility page, but the queue does not estimate financial return. The reviewer should compare the expected value of improvement with the time and resources required.

The following should NOT be automated by this playbook:

- Automatically publishing rewritten content.
- Automatically deleting or redirecting pages.
- Automatically changing search intent.
- Automatically declaring a page low quality.
- Automatically making factual or sensitive claims.
- Automatically prioritizing expensive work without human review.
- Treating the proxy label as ground truth.
- Treating feature importance as causal evidence.
- Treating the queue score as proof that a refresh will improve performance.

The system recommends what to review. A person remains responsible for the final content decision.

In [3]:
# Check that every queue item has the information needed
# for human review.

required_review_fields = [
    "content_hash_id",
    "priority",
    "reason_code",
    "archetype",
    "priority_score",
    "recommended_action"
]

missing_fields = [
    column
    for column in required_review_fields
    if column not in action_queue.columns
]

missing_values = (
    action_queue[required_review_fields]
    .isna()
    .sum()
)

print("Missing required columns:", missing_fields)

print("\nMissing values in review fields:")
display(missing_values)

assert not missing_fields, "Required review fields are missing."

assert (
    action_queue[required_review_fields]
    .isna()
    .sum()
    .sum() == 0
), "Review queue contains missing required values."

print("Human-review queue check: PASS")

Missing required columns: []

Missing values in review fields:


,0
content_hash_id,0
priority,0
reason_code,0
archetype,0
priority_score,0
recommended_action,0


Human-review queue check: PASS


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The recommendations should be reviewed if the underlying data or observed performance changes.

Light monitoring should check:

- Whether the distribution of impressions, clicks, ranking position, and engagement changes substantially.
- Whether the proportion of items receiving each reason code changes substantially.
- Whether the Week-4 baseline continues to provide useful prioritization compared with the ML alternative.
- Whether later validation periods produce materially different precision, recall, or F1 results.
- Whether analytics availability changes enough to affect the signals used by the playbook.

A revalidation or retraining review should be triggered when the monitored data show sustained distribution changes, when the baseline no longer provides useful prioritization, or when new reliable outcome labels become available.

The system should not be retrained simply because a new month of data exists. Revalidation should be driven by evidence that the current methodology may no longer represent the decision problem.

Because the current refresh label is a proxy, future work should also consider validating the queue against human-reviewed refresh outcomes before treating it as a stronger decision system.

In [4]:
# Record the current March reference distributions.
# These values can be compared with future periods during revalidation.

monitoring_reference = {
    "rows": int(len(content_df)),
    "content_items": int(content_df["content_hash_id"].nunique()),
    "clients": int(content_df["client_hash_id"].nunique()),
    "median_impressions": float(
        content_df["total_impressions"].median()
    ),
    "median_clicks": float(
        content_df["total_clicks"].median()
    ),
    "median_position": float(
        content_df["avg_position"].median()
    ),
    "baseline_opportunity_rate": float(
        content_df["baseline_opportunity"].mean()
    ),
    "high_priority_rate": float(
        (content_df["priority"] == "HIGH").mean()
    )
}

print("March 2026 monitoring reference:")

for key, value in monitoring_reference.items():
    print(f"{key}: {value}")

March 2026 monitoring reference:
rows: 331437
content_items: 331437
clients: 55
median_impressions: 2.0
median_clicks: 0.0
median_position: 8.505295944799109
baseline_opportunity_rate: 0.16290878809547515
high_priority_rate: 0.16290878809547515


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked queue is exported to `work/outputs/` so that the paper can reuse the same recommendations generated by this notebook.

The exported queue contains the ranking, priority, reason code, performance archetype, supporting metrics, and recommended human action.

The queue contains pseudonymous content identifiers only. No client names, URLs, or private queries are included.

The notebook also exports a compact monitoring summary that records the reference statistics used when the playbook was created.

In [5]:
# Create the required output directory.
os.makedirs("work/outputs", exist_ok=True)

# Export the ranked action queue.
queue_export = action_queue[
    [
        "rank",
        "content_hash_id",
        "priority",
        "reason_code",
        "archetype",
        "priority_score",
        "total_impressions",
        "total_clicks",
        "observed_ctr",
        "avg_position",
        "total_pageviews",
        "total_users",
        "total_engagement_sec",
        "total_scroll_events",
        "active_days",
        "recommended_action"
    ]
].copy()

queue_path = "work/outputs/w07_ranked_action_queue.csv"

queue_export.to_csv(
    queue_path,
    index=False
)

# Export monitoring reference values for the paper.
monitoring_path = "work/outputs/w07_monitoring_reference.json"

import json

with open(monitoring_path, "w") as f:
    json.dump(
        monitoring_reference,
        f,
        indent=2
    )

print("Queue exported:", queue_path)
print("Monitoring reference exported:", monitoring_path)
print("Queue rows exported:", len(queue_export))
print("Export columns:", list(queue_export.columns))

Queue exported: work/outputs/w07_ranked_action_queue.csv
Monitoring reference exported: work/outputs/w07_monitoring_reference.json
Queue rows exported: 331437
Export columns: ['rank', 'content_hash_id', 'priority', 'reason_code', 'archetype', 'priority_score', 'total_impressions', 'total_clicks', 'observed_ctr', 'avg_position', 'total_pageviews', 'total_users', 'total_engagement_sec', 'total_scroll_events', 'active_days', 'recommended_action']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.